
# Numerical Optimization for the Laminar Matroid Upper Bound

In the paper, we show an upper bound of $\min_{0<\beta<1}\frac{1}{\beta\Gamma(\beta)}$ on the competitive ratio of a non-adaptive algorithm for laminar matroid prophet inequalities. This notebook demonstrates the numeric details of optimizing over $\beta$ and eventually obtaining an upper bound of $6.311$.

We recall the definitions from the paper:

$$
\Gamma(\beta)=\prod_{r\geq 1}\left(1-\zeta_r(\beta r)\right) ~,
$$

where $\zeta$ is defined as follows: for $\lambda\geq 0$, let $P_\lambda\sim\mathrm{Pois}(\lambda)$, and define

$$
    \mu_k(\lambda):=\mathbb{E}[\min\{P_\lambda,k\}] ~,
$$

For $m\in[0,k)$, let $\lambda_k(m)$ be the unique value satisfying $\mu_k(\lambda_k(m))=m$, and set $\lambda_k(k)=\infty$. Then

$$
    \zeta_k(m):=\Pr[P_{\lambda_k(m)}\geq k] ~.
$$

## Computing $\zeta_k(m)$

First, for a fixed value of $\lambda$, we compute $\mu_k(\lambda):=\mathbb{E}[\min\{\mathrm{Pois}(\lambda),k\}]$
using the identity
$$
\mu_k(\lambda) = \lambda\Pr[\mathrm{Pois}(\lambda)\leq k-2]+k\Pr[\mathrm{Pois}(\lambda)\geq k] ~.
$$
Here the Poisson CDF is evaluated using `scipy.special.gammaincc`: for integer $k\geq 0$,
$$
\Pr[\mathrm{Pois}(\lambda)\leq k] = \texttt{gammaincc}(k+1,\lambda) ~.
$$

Second, using the fact that $\mu_k(\lambda)$ is monotone increasing in $\lambda$, we obtain $\lambda_k(m)$ by binary search, and return
$$
\zeta_k(m)=\Pr[\mathrm{Pois}(\lambda_k(m))\geq k] =  1-\Pr[\mathrm{Pois}(\lambda_k(m))\leq k-1] ~.
$$

In [ ]:
import math
from scipy.special import gammaincc

def poisson_cdf(lam, k):
    """Return Pr[Pois(lam) <= k]"""
    if k < 0:
        return 0.0
    return min(1.0, max(0.0, float(gammaincc(k + 1, lam))))

def poisson_tail(lam, k):
    """Return Pr[Pois(lam) >= k]"""
    return max(0.0, 1.0 - poisson_cdf(lam, k - 1))

def poisson_capped_mean(k, lam):
    """Return E[min{Pois(lam), k}]"""
    return lam * poisson_cdf(lam, k - 2) + k * poisson_tail(lam, k)

def zeta(k, m, iters=100):
    """Return zeta_k(m)"""
    if m <= 0.0:
        return 0.0
    if m >= k:
        return 1.0

    lo, hi = 0.0, max(1.0, m + 1.0)
    while poisson_capped_mean(k, hi) < m:
        hi *= 2.0

    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if poisson_capped_mean(k, mid) < m:
            lo = mid
        else:
            hi = mid

    lam = 0.5 * (lo + hi)
    return poisson_tail(lam, k)

## Optimizing the finite product

The infinite product defining $\Gamma(\beta)$ converges quickly, so in order to search for a good enough estimate of the minimizer, we evaluate $\Gamma(\beta)$ using the first $R=100$ terms of the product:
$$
\Gamma_R(\beta) = \prod_{r=1}^{100} (1-\zeta_r(\beta r)) ~.
$$

We use a coarse grid search followed by repeated local grid refinements to obtain the (near)optimal $\beta$.


In [ ]:
R=100
def objective_truncated(beta):
    """Return an estimate of Gamma(beta), truncated at the first R terms"""
    prod = 1.0
    for r in range(1, R + 1):
        prod *= 1.0 - zeta(r, beta * r)
    return beta * prod

def grid_search(f, lo, hi, num_points):
    """Maximize f over an evenly spaced grid on [lo,hi]."""
    best_x = lo
    best_value = f(lo)
    step = (hi - lo) / num_points
    for idx in range(1, num_points + 1):
        x = lo + idx * step
        value = f(x)
        if value > best_value:
            best_x = x
            best_value = value
    return best_x, best_value

def local_refinement(f, center, radius, rounds=10, num_points=100):
    """Repeatedly apply grid search on a shrinking range"""
    best_x = center
    for _ in range(rounds):
        lo = max(0.0, best_x - radius)
        hi = min(1.0, best_x + radius)
        best_x, best_value = grid_search(f, lo, hi, num_points)
        radius *= 0.01
    return best_x, best_value


beta_star, value_star = local_refinement(
    objective_truncated,
    center=0.5,
    radius=1.0,
    rounds=10,
    num_points=100,
)

print("beta* ≈", beta_star)
print("beta*Gamma_R(beta*) ≈", value_star)
print("1/(beta*Gamma_R(beta*)) ≈", 1.0 / value_star)

beta* ≈ 0.27884128613000064
beta*Gamma_R(beta*) ≈ 0.1584561443398126
1/(beta*Gamma_R(beta*)) ≈ 6.310894438119601


## Bounding the Infinite Product

Finally we show that for $\beta^*\approx 0.27884$, the gap between $\Gamma_R(\beta^*)$ and $\Gamma(\beta^*)$ is bounded and sufficiently small. Since
$$
\Gamma(\beta) = \Gamma_R(\beta)\cdot \prod_{r>R}(1-\zeta_r(\beta r))  \geq  \Gamma_R(\beta) \cdot \left(1-\sum_{r>R}\zeta_r(\beta r)\right) ~,
$$
it suffices to upper bound $\sum_{r>R}\zeta_r(\beta r)$.

For every $r\geq 101$, let $\lambda_r(\beta^* r)$ be the parameter that defines $\zeta_r(\beta^* r)$, i.e.
$$
\mathbb{E}[\min\{\mathrm{Pois}(\lambda_r(\beta^* r)),r\}]= \beta^* r ~.
$$
We claim that $\lambda_r(\beta^* r) \leq \gamma r$ where $\gamma = 0.279$, slightly larger than $\beta^*$. Indeed,
$$
\mathbb{E}[\min\{\mathrm{Pois}(\gamma r),r\}] = \gamma r-\mathbb{E}[(\mathrm{Pois}(\gamma r)-r)^+] \geq \gamma r(1 - \Pr[\mathrm{Pois}(\gamma r) \geq r]) ~.
$$

By the Chernoff bound,
$$
\Pr[\mathrm{Pois}(\gamma r)\geq r] \leq \left(\gamma e^{1-\gamma}\right)^r := a^r ~,
$$
Hence for $r\geq 101$, we have
$$
\mathbb{E}[\min\{\mathrm{Pois}(\gamma r),r\}] \geq \gamma r(1-a^r) \geq \beta^* r ~.
$$
So by monotonicity of $\mu_r$ we conclude that $\lambda_r(\beta^* r) \leq \gamma r$. Therefore,
$$
\zeta_r(\beta^* r) \leq \Pr[\mathrm{Pois}(\gamma r)\geq r] \leq a^r ~.
$$

This implies
$$
\sum_{r>100}\zeta_r(\beta^* r) \leq \sum_{r=101}^{\infty}a^r = \frac{a^{101}}{1-a} \leq 2\times 10^{-24} ~.
$$
Thus, the competitive ratio at $\beta^*$ is at most
$$
\frac{1}{\beta^*\Gamma(\beta^*)} \leq \frac{1}{\beta^* \Gamma_R(\beta^*)\cdot (1-2\times 10^{-24})} \leq 6.311 ~.
$$

The following code certifies the convergence of $\frac{1}{\beta^* \Gamma_R(\beta^*)}$.

In [8]:
def truncated_ratio(beta, R):
    prod = 1.0
    for r in range(1, R + 1):
        prod *= 1.0 - zeta(r, beta * r)
    return 1.0 / (beta * prod)

for R in [10, 50, 100, 200, 500, 1000]:
    print("R = ", R, truncated_ratio(beta_star, R))

R =  10 6.3059378696767885
R =  50 6.3108944381190595
R =  100 6.310894438119601
R =  200 6.310894438119601
R =  500 6.310894438119601
R =  1000 6.310894438119601
